In [ ]:
import re
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

v4_root = Path("C:\D\GazProm\nogit\Digital_core_v4")
output_csv = Path("C:\D\GazProm\nogit\Digital_core_v5.2/pairs_full_unique.csv")
output_csv.parent.mkdir(parents=True, exist_ok=True)

TOLERANCE = 0.02

pattern = re.compile(r'(.+)_(\d+\.\d+)_(\d+\.\d+)_frag\d+\.jpg')

# Сбор всех файлов
all_files = []
for field_dir in tqdm(list(v4_root.iterdir()), desc="Сканирование"):
    if not field_dir.is_dir():
        continue
    for light in ["ДС", "УФ"]:
        light_dir = field_dir / light
        if not light_dir.exists():
            continue
        for f in light_dir.glob("*.jpg"):
            m = pattern.match(f.name)
            if m:
                all_files.append({
                    'field': field_dir.name,
                    'light': light,
                    'mineral': m.group(1),
                    'd_from': float(m.group(2)),
                    'd_to': float(m.group(3)),
                    'path': str(f.absolute())
                })

# Группировка
groups = defaultdict(list)
for item in all_files:
    key = (item['field'], item['mineral'], item['d_from'])
    groups[key].append(item)

# Построение уникальных пар (один УФ – один ДС)
used_uv = set()
unique_pairs = []

for key, items in tqdm(groups.items(), desc="Поиск уникальных пар"):
    ds_items = [i for i in items if i['light'] == "ДС"]
    uv_items = [i for i in items if i['light'] == "УФ" and i['path'] not in used_uv]
    if not ds_items or not uv_items:
        continue
    # Для каждого ДС ищем лучший УФ (минимальная разница d_to)
    for ds in ds_items:
        best_uv = None
        best_diff = TOLERANCE
        for uv in uv_items:
            diff = abs(ds['d_to'] - uv['d_to'])
            if diff < best_diff:
                best_diff = diff
                best_uv = uv
        if best_uv:
            unique_pairs.append({
                'ds_path': ds['path'],
                'uv_path': best_uv['path'],
                'mineral': ds['mineral']
            })
            used_uv.add(best_uv['path'])
            # удаляем использованный УФ из списка (чтобы не использовать повторно)
            uv_items = [uv for uv in uv_items if uv['path'] not in used_uv]

df = pd.DataFrame(unique_pairs)
df.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f"Создано {len(df)} уникальных пар (каждый УФ уникален)")

In [ ]:
import pandas as pd

# Загружаем уникальные пары
df = pd.read_csv("C:\D\GazProm\nogit\Digital_core_v5.2/pairs_full_unique.csv")

# Словарь для объединения в 7 классов
class_mapping = {
    # Песчаники
    'Песчаник': 'Песчаник',
    'Песчаник_с_прослоями_алевролита': 'Песчаник',
    'Песчаник_карбонатный': 'Песчаник',
    'Песчаник_с_прослоями_алевролита_и_аргиллита': 'Песчаник',
    'Песчаник_с_прослоями_аргиллита': 'Песчаник',
    'Песчаник_с_включениями_алевролита_и_аргиллита': 'Песчаник',
    'Песчаник_с_включениями_угля': 'Песчаник',
    
    # Аргиллиты
    'Аргиллит': 'Аргиллит',
    'Аргиллит_с_прослоями_алевролита': 'Аргиллит',
    'Аргиллит_с_прослоями_песчаника_и_алевролита': 'Аргиллит',
    'Аргиллит_с_прослоями_песчаника': 'Аргиллит',
    'Аргиллит_с_включениями_угля': 'Аргиллит',
    'Аргиллит_алевритовый': 'Аргиллит',
    'Аргиллит_углистый': 'Аргиллит',
    
    # Алевролиты
    'Алевролит': 'Алевролит',
    'Алевролит_с_прослоями_песчаника': 'Алевролит',
    'Алевролит_с_прослоями_песчаника_и_аргиллита': 'Алевролит',
    'Алевролит_карбонатный': 'Алевролит',
    'Алевролит_с_прослоями_аргиллита': 'Алевролит',
    'Алевролит_глинистый': 'Алевролит',
    'Алевролит_с_включениями_угля': 'Алевролит',
    'Алевролит_песчанистый': 'Алевролит',
    
    # Переслаивания
    'Переслаивание_песчаника,_аргиллита_и_алевролита': 'Переслаивание',
    'Переслаивание_аргиллита_и_алевролита': 'Переслаивание',
    'Переслаивание_песчаника_и_аргиллита': 'Переслаивание',
    'Переслаивание_песчаника_и_алевролита': 'Переслаивание',
    'Чередование_аргиллита,_алевролита_и_песчаника': 'Переслаивание',
    
    # Глины
    'Глина_аргиллитоподобная': 'Глина',
    'Глина_опоковидная_с_включением_глинистых_опок': 'Глина',
    'Глина_аргиллитоподобная_с_прослоями_глины_опоковидной': 'Глина',
    'Глина_опоковидная': 'Глина',
    'Опока_глинистая': 'Глина',
    
    # Угли
    'Уголь': 'Углистые породы',
    'Уголь_с_прослоями_аргиллита': 'Углистые породы',
    
    # Прочие
    'Глинисто-карбонатная_порода': 'Прочие',
    'Известняк': 'Прочие',
    'Породы_фундамента': 'Прочие',
}

# Применяем маппинг
df['mineral_grouped'] = df['mineral'].map(class_mapping)

# Проверяем, все ли классы сопоставлены
missing = df[df['mineral_grouped'].isna()]['mineral'].unique()
if len(missing) > 0:
    print(f"Внимание! Не сопоставлены: {missing}")
else:
    print("Все классы успешно сопоставлены.")

# Сохраняем результат
output_path = "C:\D\GazProm\nogit\Digital_core_v5.2/pairs_7classes.csv"
df.to_csv(output_path, index=False, encoding='utf-8-sig')

# Баланс классов
print("\nБАЛАНС КЛАССОВ (количество пар):")
print(df['mineral_grouped'].value_counts())

In [ ]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import random
from PIL import Image

def load_image(path):
    with Image.open(path) as img:
        return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

def save_image(path, img):
    img = np.clip(img, 0, 255).astype(np.uint8)
    Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)).save(str(path), quality=90)

def augment_pair(ds_img, uv_img):
    """Применяет одинаковые аугментации к паре (ДС, УФ)"""
    # Вертикальное отражение (зеркало по Y)
    if random.random() > 0.5:
        ds_img = cv2.flip(ds_img, 0)
        uv_img = cv2.flip(uv_img, 0)
    
    # Изменение яркости (±20%)
    if random.random() > 0.5:
        alpha = 0.8 + random.random() * 0.4
        ds_img = cv2.convertScaleAbs(ds_img, alpha=alpha)
        uv_img = cv2.convertScaleAbs(uv_img, alpha=alpha)
    
    # Изменение контраста (CLAHE) – отдельно для каждого канала
    if random.random() > 0.5:
        for img in [ds_img, uv_img]:
            lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
            l, a, b = cv2.split(lab)
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            l = clahe.apply(l)
            lab = cv2.merge((l, a, b))
            img[:] = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    
    # Горизонтальный сдвиг (±5%)
    if random.random() > 0.5:
        h, w = ds_img.shape[:2]
        shift = int(w * 0.05 * (random.random() - 0.5))
        M = np.float32([[1, 0, shift], [0, 1, 0]])
        ds_img = cv2.warpAffine(ds_img, M, (w, h))
        uv_img = cv2.warpAffine(uv_img, M, (w, h))
    
    return ds_img, uv_img

# Загрузка исходного CSV
df = pd.read_csv("C:\D\GazProm\nogit\Digital_core_v5.2/pairs_7classes.csv")
target = 5000
classes_to_augment = ['Глина', 'Прочие', 'Углистые породы']

# Папка для аугментированных пар
augmented_dir = Path("C:\D\GazProm\nogit\Digital_core_v5.2/augmented_pairs")
augmented_dir.mkdir(parents=True, exist_ok=True)

new_pairs = []

for class_name in classes_to_augment:
    class_df = df[df['mineral_grouped'] == class_name]
    current_count = len(class_df)
    needed = target - current_count
    if needed <= 0:
        print(f"{class_name}: уже {current_count} (>= {target})")
        continue
    
    print(f"{class_name}: {current_count} -> нужно +{needed}")
    # Создаём копии существующих пар
    for i in tqdm(range(needed), desc=f"Аугментация {class_name}"):
        # Выбираем случайную исходную пару
        row = class_df.sample(1).iloc[0]
        ds_img = load_image(row['ds_path'])
        uv_img = load_image(row['uv_path'])
        
        # Применяем аугментацию
        ds_aug, uv_aug = augment_pair(ds_img, uv_img)
        
        # Сохраняем аугментированные изображения
        new_ds_name = f"{Path(row['ds_path']).stem}_aug_{i:04d}.jpg"
        new_uv_name = f"{Path(row['uv_path']).stem}_aug_{i:04d}.jpg"
        ds_aug_path = augmented_dir / new_ds_name
        uv_aug_path = augmented_dir / new_uv_name
        save_image(ds_aug_path, ds_aug)
        save_image(uv_aug_path, uv_aug)
        
        # Добавляем запись в новый CSV
        new_pairs.append({
            'ds_path': str(ds_aug_path.absolute()),
            'uv_path': str(uv_aug_path.absolute()),
            'mineral_grouped': class_name
        })

# Объединяем старые и новые пары
df_augmented = pd.concat([df, pd.DataFrame(new_pairs)], ignore_index=True)
output_path = "C:\D\GazProm\nogit\Digital_core_v5.2/pairs_7classes_augmented.csv"
df_augmented.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"\nСохранено {len(df_augmented)} пар в {output_path}")
print("\nБАЛАНС КЛАССОВ ПОСЛЕ АУГМЕНТАЦИИ:")
print(df_augmented['mineral_grouped'].value_counts())

In [ ]:
import pandas as pd
import shutil
from pathlib import Path
import random

# Загружаем датасет
df = pd.read_csv("C:\D\GazProm\nogit\Digital_core_v5.2/pairs_7classes_augmented.csv")

# Убедимся, что колонка с классом называется 'mineral_grouped'
if 'mineral_grouped' not in df.columns:
    raise ValueError("Нет колонки 'mineral_grouped'")

# Создаём папки для ручной проверки
manual_dir = Path("C:\D\GazProm\nogit\Digital_core_v5.2/manual_check")
manual_dir.mkdir(parents=True, exist_ok=True)

# Собираем пары для ручной проверки (по 2 из каждого класса)
manual_pairs = []
train_val_pairs = []

for class_name in df['mineral_grouped'].unique():
    class_df = df[df['mineral_grouped'] == class_name]
    # Берём 2 случайные пары
    if len(class_df) >= 2:
        sample = class_df.sample(2, random_state=42)
        manual_pairs.append(sample)
        # Остальные идут в train/val
        train_val_pairs.append(class_df.drop(sample.index))
    else:
        # Если в классе меньше 2 пар, берём все (но такое вряд ли)
        manual_pairs.append(class_df)
        print(f"Внимание: в классе {class_name} всего {len(class_df)} пар")

# Объединяем
manual_df = pd.concat(manual_pairs)
train_val_df = pd.concat(train_val_pairs)

# Сохраняем отдельные CSV
manual_df.to_csv("C:\D\GazProm\nogit\Digital_core_v5.2/manual_pairs.csv", index=False)
train_val_df.to_csv("C:\D\GazProm\nogit\Digital_core_v5.2/train_val_pairs.csv", index=False)

print(f"В ручную проверку отложено {len(manual_df)} пар (по 2 из каждого класса)")
print(f"Осталось для train/val: {len(train_val_df)} пар")

# Копируем изображения для ручной проверки в отдельную папку
for idx, row in manual_df.iterrows():
    ds_src = Path(row['ds_path'])
    uv_src = Path(row['uv_path'])
    # Создаём подпапку для класса
    class_dir = manual_dir / row['mineral_grouped']
    class_dir.mkdir(parents=True, exist_ok=True)
    # Копируем
    shutil.copy2(ds_src, class_dir / f"ds_{row['mineral_grouped']}_{idx}.jpg")
    shutil.copy2(uv_src, class_dir / f"uv_{row['mineral_grouped']}_{idx}.jpg")

print(f"Изображения для ручной проверки скопированы в {manual_dir}")

# Теперь train_val_df можно разбить на train и val (стратифицированно)
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    train_val_df, 
    test_size=0.2, 
    stratify=train_val_df['mineral_grouped'], 
    random_state=42
)

train_df.to_csv("C:\D\GazProm\nogit\Digital_core_v5.2/train_pairs.csv", index=False)
val_df.to_csv("C:\D\GazProm\nogit\Digital_core_v5.2/val_pairs.csv", index=False)

print(f"Train: {len(train_df)} пар")
print(f"Val: {len(val_df)} пар")
print("\nРаспределение классов в train:")
print(train_df['mineral_grouped'].value_counts())
print("\nРаспределение классов в val:")
print(val_df['mineral_grouped'].value_counts())